# 10 - Production Comparison

Compare Soil State Transformer against production XGBoost ensemble.

## Evaluation Goals
1. Compare accuracy on held-out test data
2. Evaluate on production field data from `your-eval-bucket`
3. Assess computational efficiency
4. Make go/no-go recommendation for production integration

In [ ]:
import sys
sys.path.insert(0, '../..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json
import time

from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

DATA_DIR = Path('../../data/processed')
RESULTS_DIR = Path('../../results/evaluations')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Production comparison bucket (read-only)
PRODUCTION_BUCKET = 'your-eval-bucket'

## 1. Load Models

In [ ]:
# Load baseline results (from notebook 05)
baseline_path = Path('../../results/model_experiments/baseline_summary.json')

if baseline_path.exists():
    with open(baseline_path) as f:
        baseline_summary = json.load(f)
    print("Baseline metrics loaded:")
    for target, metrics in baseline_summary.get('baseline_metrics', {}).items():
        print(f"  {target}: R²={metrics['val_r2']:.3f}")
else:
    print("Run notebook 05_baseline_models.ipynb first")
    baseline_summary = None

In [ ]:
# Load SST model (would be trained in separate script)
sst_path = Path('../../models/pretrained/sst_v1/checkpoint_best.pt')

if sst_path.exists():
    import torch
    sst_model = torch.load(sst_path)
    print("SST model loaded")
else:
    print(f"SST model not found at {sst_path}")
    print("Train the model first using training scripts")
    sst_model = None

## 2. Load Test Data

In [ ]:
# Load test set
test_path = DATA_DIR / 'test.parquet'

if test_path.exists():
    test_df = pd.read_parquet(test_path)
    print(f"Test set: {len(test_df):,} samples")
else:
    print("Run notebook 04_data_integration.ipynb first")
    test_df = None

## 3. Comparison Framework

In [ ]:
def compare_models(y_true, predictions_dict):
    """
    Compare multiple model predictions.
    
    Args:
        y_true: Ground truth values
        predictions_dict: {model_name: predictions}
    
    Returns:
        Comparison DataFrame
    """
    results = []
    
    for model_name, y_pred in predictions_dict.items():
        results.append({
            'model': model_name,
            'r2': r2_score(y_true, y_pred),
            'rmse': np.sqrt(mean_squared_error(y_true, y_pred)),
            'mae': mean_absolute_error(y_true, y_pred),
            'bias': np.mean(y_pred - y_true)
        })
    
    return pd.DataFrame(results)

In [ ]:
def plot_comparison(y_true, predictions_dict, title="Model Comparison"):
    """Visualize prediction quality."""
    n_models = len(predictions_dict)
    fig, axes = plt.subplots(1, n_models, figsize=(5*n_models, 5))
    
    if n_models == 1:
        axes = [axes]
    
    for ax, (name, y_pred) in zip(axes, predictions_dict.items()):
        ax.scatter(y_true, y_pred, alpha=0.5, s=10)
        
        # Perfect prediction line
        lims = [min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())]
        ax.plot(lims, lims, 'r--', alpha=0.8, label='Perfect')
        
        # Metrics
        r2 = r2_score(y_true, y_pred)
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        
        ax.set_xlabel('Actual')
        ax.set_ylabel('Predicted')
        ax.set_title(f'{name}\nR²={r2:.3f}, RMSE={rmse:.2f}')
        ax.legend()
    
    plt.suptitle(title)
    plt.tight_layout()
    return fig

## 4. Performance Comparison

In [ ]:
# Example comparison structure
TARGETS = ['SOC', 'pH', 'clay', 'sand', 'N', 'CEC']

comparison_results = []

for target in TARGETS:
    print(f"\n{'='*50}")
    print(f"Target: {target}")
    print('='*50)
    
    # Placeholder for actual evaluation
    # In practice, load predictions from each model
    
    if baseline_summary:
        baseline_r2 = baseline_summary.get('baseline_metrics', {}).get(target, {}).get('val_r2', None)
        if baseline_r2:
            print(f"  Baseline R²: {baseline_r2:.3f}")
    
    # SST comparison would go here
    print(f"  SST R²: [Run training first]")

## 5. Computational Efficiency

In [ ]:
def benchmark_inference(model, test_data, n_runs=10):
    """
    Benchmark model inference speed.
    """
    times = []
    
    for _ in range(n_runs):
        start = time.time()
        _ = model.predict(test_data)
        times.append(time.time() - start)
    
    return {
        'mean_time': np.mean(times),
        'std_time': np.std(times),
        'samples_per_second': len(test_data) / np.mean(times)
    }

print("Benchmark function ready")
print("\nExpected performance:")
print("  XGBoost: ~10,000 samples/second")
print("  SST (CPU): ~1,000 samples/second")
print("  SST (GPU): ~10,000 samples/second")

## 6. Go/No-Go Decision Framework

In [ ]:
def make_recommendation(comparison_results):
    """
    Make go/no-go recommendation for production.
    
    Criteria:
    - SST R² should exceed baseline by >5%
    - RMSE should be lower
    - Inference time should be acceptable (<100ms per sample)
    """
    
    recommendation = {
        'decision': 'PENDING',
        'rationale': [],
        'next_steps': []
    }
    
    # Placeholder logic
    recommendation['rationale'].append('Awaiting full evaluation')
    recommendation['next_steps'].append('Complete model training')
    recommendation['next_steps'].append('Run full comparison')
    
    return recommendation

rec = make_recommendation(comparison_results)
print("\n" + "="*50)
print("RECOMMENDATION")
print("="*50)
print(f"Decision: {rec['decision']}")
print("\nRationale:")
for r in rec['rationale']:
    print(f"  - {r}")
print("\nNext Steps:")
for s in rec['next_steps']:
    print(f"  - {s}")

## 7. Save Evaluation Report

In [ ]:
report = {
    'evaluation_date': pd.Timestamp.now().isoformat(),
    'models_compared': ['XGBoost Ensemble (Baseline)', 'Soil State Transformer'],
    'targets': TARGETS,
    'recommendation': rec,
    'notes': [
        'Complete model training before final evaluation',
        'Test on production data from your-eval-bucket',
        'Consider field-specific LoRA adapters for deployment'
    ]
}

with open(RESULTS_DIR / 'production_comparison_report.json', 'w') as f:
    json.dump(report, f, indent=2)

print(f"\nReport saved to {RESULTS_DIR / 'production_comparison_report.json'}")